# Canary-Qwen Batch Evaluation

This notebook evaluates the NVIDIA Canary-Qwen-2.5B model on a dataset of audio segments.
It uses the NeMo framework directly for batch inference.

In [ ]:
import os
import json
import torch
from google.cloud import storage
from nemo.collections.speechlm2.models import SALM

# Import common GCS utils and runner
from common.gcs_utils import (
    parse_gcs_uri,
    download_blob_to_file,
    download_jsonl_manifest,
    upload_inference_results,
)
from common.inference_pipeline_runner import run_inference_pipeline

from common.audio_utils import preprocess_audio_for_model

# Configure logging
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
# Configuration
GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"
MODEL_NAME = "nvidia/canary-qwen-2.5b"
SELECTED_MODEL_KEY = "canary-qwen-2.5b"
BATCH_SIZE = 4  # Smaller batch size for safety
LIMIT = 10  # For testing

In [ ]:
print("Loading NVIDIA Canary-Qwen-2.5B into GPU VRAM...")
model = SALM.from_pretrained(MODEL_NAME).half().eval().to("cuda")

In [ ]:
# @title Define helper functions and Run Evaluation


def prompt_formatter(entry, local_path):
    dialog = [
        {
            "role": "user",
            # Prime the LLM to expect noisy, dispatch-specific vocabulary
            "content": "Transcribe the following emergency dispatch radio traffic: <|audioplaceholder|>",
            "audio": [local_path],
        }
    ]
    return (dialog, local_path)


def canary_inference(model, prompts):
    dialogs = [p[0] for p in prompts]

    answer_ids = model.generate(
        prompts=dialogs,
        max_new_tokens=128,
        do_sample=False,  # Greedy decoding (stops it from trying to "chat" back)
        repetition_penalty=1.0,  # Lowered slightly from 1.2 so we don't accidentally penalize short, valid phrases
    )

    return answer_ids


def result_decoder(ans, model):
    token_ids = ans.cpu().tolist()
    text = model.tokenizer.ids_to_text(token_ids)

    # Safely extract just the assistant's response using the Qwen chat template tags
    # This avoids accidentally splitting on common English words.
    if "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1]
    elif "<|imstart|>assistant" in text:
        text = text.split("<|imstart|>assistant")[-1]

    # Strip out the end-of-turn token if it's there
    text = text.replace("<|im_end|>", "").replace("<|imend|>", "")

    return text.strip()


storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_inference_pipeline(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=canary_inference,
    preprocess_fn=preprocess_audio_for_model,
    decode_fn=result_decoder,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
)

In [ ]:
# Upload results directly to GCS from memory (uncomment to use)
gcs_uri = upload_inference_results(
    storage_client,
    GCS_BUCKET,
    PROJECT_NAME,
    SELECTED_MODEL_KEY,
    EXPERIMENT_NAME,
    results_list,
)